# Abundant 25x20: no regularization vs Gini


In [1]:
from __future__ import annotations

import contextlib
import hashlib
import inspect
import io
import json
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 280)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing opinion_dynamics/.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rl_envs_forge.envs.network_graph.network_graph import NetworkGraph
from rl_envs_forge.envs.network_graph.graph_utils import (
    compute_eigenvector_centrality,
    compute_laplacian,
)
from opinion_dynamics.baseline import centrality_based_continuous_control
from opinion_dynamics.identify_nonlinear_unbounded_regularized import (
    GraphIdentifierEnv,
    pairs_from_intermediate,
    train_graph_identifier,
)
import opinion_dynamics.experiments.online_single_shot_unbounded as online_single_shot

try:
    GIT_COMMIT = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception:
    GIT_COMMIT = "unknown"

print("Repository root:", REPO_ROOT)
print("Git commit:", GIT_COMMIT)
print("identifier:", inspect.getsourcefile(GraphIdentifierEnv))
print("online_single_shot:", inspect.getsourcefile(online_single_shot))
print("NetworkGraph:", inspect.getsourcefile(NetworkGraph))


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_118648\1508296264.py:18: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Repository root: d:\Work\repos\RL\unknown_graph_networks
Git commit: 0eca8fe
identifier: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\identify_nonlinear_unbounded_regularized.py
online_single_shot: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\online_single_shot_unbounded.py
NetworkGraph: c:\Users\Chainsword\anaconda3\envs\phd_rl_algos\Lib\site-packages\rl_envs_forge\envs\network_graph\network_graph.py


## Configuration


In [2]:
# Parallel worker mode: keep the same eight-way environment sharding as the
# final single-shot broad benchmark.
NUM_SHARDS = 8
SHARD_ENV = "ABUNDANT_FULL_SHARD_ID"
THREAD_ENV = "ABUNDANT_FULL_TORCH_THREADS"

_shard_value = os.environ.get(SHARD_ENV)
WORKER_MODE = _shard_value is not None
SHARD_ID = int(_shard_value) if WORKER_MODE else None
if WORKER_MODE and SHARD_ID not in range(NUM_SHARDS):
    raise ValueError(f"{SHARD_ENV} must be in 0..{NUM_SHARDS - 1}; got {SHARD_ID}")

TORCH_THREADS = int(os.environ.get(THREAD_ENV, "2"))
torch.set_num_threads(max(1, TORCH_THREADS))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

DEVICE = os.environ.get("ABUNDANT_FULL_DEVICE", "cpu").lower()
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable.")

STUDY_NAME = "abundant_matched_distribution_25x20_none_vs_gini_2026_10_02"
PIPELINE_VERSION = "2026-10-02-abundant-25x20-none-vs-gini-v1"

RESULTS_ROOT = REPO_ROOT / "opinion_dynamics" / "experiments" / "results"

# Exact evaluation settings from the final broad single-shot benchmark.
N = 15
TARGET = 1.0
NUM_CAMPAIGNS = 20
T_CAMPAIGN = 0.5
T_S = 0.1
MAX_U = 0.20
TOTAL_CONTROLLED_BUDGET = 6.0
B_CAMPAIGN = TOTAL_CONTROLLED_BUDGET / (NUM_CAMPAIGNS - 1)
LAMBDA_MIX = 1.00

FIT_LR = 1e-3
FIT_MAX_STEPS = 1_000
FIT_MAE_STOP = 5e-4
FIT_BATCH_SIZE = 256
FIT_CHECK_EVERY = 200
IDENTIFIER_KWARGS = {"hidden_dim": 16}
LEARNING_SEEDS = [0, 1, 2]

# Paired comparison. Gini 1e-4 was the strongest abundant-data trade-off
# in the earlier sweep, so this rerun gives that regularizer a clean test
# under the corrected 25x20 collection regime.
GINI_LAMBDA = 1e-4
VARIANTS = [
    {"regularizer_kind": "none", "regularizer_lambda": 0.0},
    {"regularizer_kind": "gini", "regularizer_lambda": GINI_LAMBDA},
]

# Reuse the corrected 25x20 no-regularization run whenever it exists.
SOURCE_STUDY = "final_unregularized_abundant_matched_distribution_25x20_2026_10_02"
SOURCE_RESULTS_DIR = RESULTS_ROOT / "experiment_2026_10_02_final_unregularized_abundant_matched_distribution_25x20"
SOURCE_CACHE_STUDY_ROOT = RESULTS_ROOT / "_trial_cache" / SOURCE_STUDY

# Abundant-data axis. Keep the same total collection scale (500 campaign
# steps/environment) while matching the full 20-campaign evaluation horizon.
# This also matches the evaluation ratio of one passive first campaign followed
# by random-score water-filling campaigns within each trajectory.
ABUNDANT_RESETS = 25
ABUNDANT_CAMPAIGNS_PER_RESET = NUM_CAMPAIGNS
DATA_SEED_BASE = 1_200_000
COLLECTION_POLICY = "passive_then_random_score_waterfill"
COLLECTION_X0 = "matched_environment_distribution"

# Original generic/random benchmark seeds.
UNSTRUCTURED_TOPOLOGY_SEEDS = [3, 4, 5, 6, 7]
UNSTRUCTURED_OPINION_SEEDS = [0, 1, 2, 4, 5]
UNSTRUCTURED_DYNAMICS = {
    "laplacian": {},
    "coca": {},
    "hegselmannkrause": {"hk_epsilon": 0.50, "hk_include_self": True},
    "friedkinjohnsen": {"fj_lambda": 0.98},
    "nonlinearinfluence": {"nonlinear_beta": 4.0},
    "repulsion": {"repulsion_epsilon": 0.30, "repulsion_strength": 0.10},
}

# Structured mechanism parameters, copied from the final single-shot benchmark.
STRUCTURED_HK_EPSILON = 0.25
STRUCTURED_FJ_LAMBDA = 0.98
STRUCTURED_NLI_BETA = 4.0
STRUCTURED_ENVIRONMENT_SEEDS = [0, 1, 2, 3, 4]

HK_FAMILIES = {
    "ring_bridge": {
        "peer_pattern": "ring",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.40,
        "bridge_weight": 3.0,
    },
    "paired_bridge": {
        "peer_pattern": "paired",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.55,
        "bridge_weight": 3.0,
    },
    "dense_weak_bridge": {
        "peer_pattern": "dense",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.08,
        "bridge_weight": 3.0,
    },
}

BROADCAST_FAMILIES = {
    "asymmetric_broadcast": {"kind": "asymmetric", "hub_influence_total": 0.85},
    "dual_broadcast": {"kind": "dual", "hub_influence_total": 0.85},
    "split_broadcast": {"kind": "split", "hub_influence_total": 0.85},
}

FJ_HUB_LEVEL = 0.16
FJ_LEAF_LEVEL = 0.48
NLI_HUB_LEVEL = 0.10
NLI_LEAF_LEVEL = 0.50
EDGE_WEIGHT_JITTER_FRAC = 0.05
HK_OPINION_JITTER = 0.006
BROADCAST_OPINION_JITTER = 0.010

RESULTS_DIR = RESULTS_ROOT / "experiment_2026_10_02_abundant_matched_distribution_25x20_none_vs_gini"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = RESULTS_ROOT / "_trial_cache" / STUDY_NAME
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("Mode:", "worker" if WORKER_MODE else "analysis/check")
if WORKER_MODE:
    print("Shard:", f"{SHARD_ID + 1}/{NUM_SHARDS}")
print("Device:", DEVICE)
print("Torch threads:", torch.get_num_threads())
print("Campaign budget:", B_CAMPAIGN)
print("Abundant resets:", ABUNDANT_RESETS)
print("Collection campaigns/reset:", ABUNDANT_CAMPAIGNS_PER_RESET)


Mode: analysis/check
Device: cpu
Torch threads: 2
Campaign budget: 0.3157894736842105
Abundant resets: 25
Collection campaigns/reset: 20


## Graph helpers and the exact final single-shot topology families


In [3]:
def _normalize_rows(raw: np.ndarray) -> np.ndarray:
    raw = np.asarray(raw, dtype=float).copy()
    np.fill_diagonal(raw, 0.0)
    sums = raw.sum(axis=1, keepdims=True)
    if np.any(sums <= 0):
        raise RuntimeError("Every graph row must have positive mass.")
    A = raw / sums
    np.fill_diagonal(A, 0.0)
    return A


def _jitter_positive_edges(raw: np.ndarray, rng: np.random.Generator, frac: float) -> np.ndarray:
    out = np.asarray(raw, dtype=float).copy()
    mask = out > 0
    multipliers = rng.uniform(1.0 - float(frac), 1.0 + float(frac), size=out.shape)
    out[mask] *= multipliers[mask]
    np.fill_diagonal(out, 0.0)
    return out


def _add_undirected(raw: np.ndarray, i: int, j: int, weight: float) -> None:
    raw[i, j] += float(weight)
    raw[j, i] += float(weight)


def centrality_from_A(A: np.ndarray) -> np.ndarray:
    v = compute_eigenvector_centrality(compute_laplacian(np.asarray(A, dtype=float)))
    v = np.asarray(v, dtype=float).reshape(-1)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    if v.sum() < 0:
        v = -v
    v = np.maximum(v, 0.0)
    if v.sum() <= 1e-12:
        v = np.abs(v)
    if v.sum() <= 1e-12:
        return np.full(N, 1.0 / N)
    return v / v.sum()


def build_unstructured_adjacency(topology_seed: int) -> np.ndarray:
    x_dummy = np.linspace(0.0, 1.0, N)
    graph_env = NetworkGraph(
        num_agents=N,
        graph_model="barabasi_albert",
        ba_m=2,
        ba_prune_max_frac=0.5,
        ba_qsc_tol=1e-8,
        ba_max_tries=500,
        max_u=MAX_U,
        budget=1000.0,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        max_steps=NUM_CAMPAIGNS + 20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        seed=int(topology_seed),
        terminate_when_converged=False,
        dynamics_model="laplacian",
        initial_opinions=x_dummy,
        control_resistance=np.zeros(N, dtype=float),
    )
    return np.asarray(graph_env.connectivity_matrix, dtype=float).copy()


def build_unstructured_x0(opinion_seed: int) -> np.ndarray:
    base = np.linspace(0.0, 1.0, N)
    rng = np.random.default_rng(920_000 + int(opinion_seed))
    return rng.permutation(base).astype(float)


UNSTRUCTURED_A_BY_SEED = {
    int(seed): build_unstructured_adjacency(int(seed))
    for seed in UNSTRUCTURED_TOPOLOGY_SEEDS
}

HK_LOW_CLUSTER = tuple(range(0, 7))
HK_HIGH_CLUSTER = tuple(range(7, 15))
HK_LOW_HUB = 0
HK_HIGH_HUB = 7
BROADCAST_HUBS = (0, 1)
BROADCAST_LEAVES = tuple(range(2, N))
BROADCAST_GROUP_A = tuple(range(2, 8))
BROADCAST_GROUP_B = tuple(range(8, N))


def build_hk_graph(family_name: str, environment_seed: int) -> tuple[np.ndarray, np.ndarray]:
    params = HK_FAMILIES[family_name]
    raw = np.zeros((N, N), dtype=float)

    for node in HK_LOW_CLUSTER:
        if node != HK_LOW_HUB:
            _add_undirected(raw, HK_LOW_HUB, node, params["hub_leaf_weight"])
    for node in HK_HIGH_CLUSTER:
        if node != HK_HIGH_HUB:
            _add_undirected(raw, HK_HIGH_HUB, node, params["hub_leaf_weight"])

    for cluster, hub in [(HK_LOW_CLUSTER, HK_LOW_HUB), (HK_HIGH_CLUSTER, HK_HIGH_HUB)]:
        leaves = [node for node in cluster if node != hub]
        pattern = params["peer_pattern"]
        w = float(params["peer_weight"])
        if pattern == "ring":
            for left, right in zip(leaves, leaves[1:] + leaves[:1]):
                _add_undirected(raw, left, right, w)
        elif pattern == "paired":
            for index in range(0, len(leaves) - 1, 2):
                _add_undirected(raw, leaves[index], leaves[index + 1], w)
            if len(leaves) % 2 == 1:
                _add_undirected(raw, leaves[-1], leaves[0], 0.5 * w)
        elif pattern == "dense":
            for pos, left in enumerate(leaves):
                for right in leaves[pos + 1:]:
                    _add_undirected(raw, left, right, w)
        else:
            raise ValueError(pattern)

    _add_undirected(raw, HK_LOW_HUB, HK_HIGH_HUB, params["bridge_weight"])
    rng = np.random.default_rng(
        10_000 + 100 * list(HK_FAMILIES).index(family_name) + int(environment_seed)
    )
    raw = _jitter_positive_edges(raw, rng, EDGE_WEIGHT_JITTER_FRAC)
    return raw, _normalize_rows(raw)


def build_hk_x0(environment_seed: int) -> np.ndarray:
    base = np.array(
        [0.35, 0.27, 0.28, 0.29, 0.30, 0.31, 0.32,
         0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.70, 0.71],
        dtype=float,
    )
    rng = np.random.default_rng(20_000 + int(environment_seed))
    return np.clip(base + rng.uniform(-HK_OPINION_JITTER, HK_OPINION_JITTER, size=N), 0.0, 1.0)


def build_broadcaster_graph(family_name: str, environment_seed: int) -> tuple[np.ndarray, np.ndarray]:
    params = BROADCAST_FAMILIES[family_name]
    h = float(params["hub_influence_total"])
    kind = params["kind"]
    raw = np.zeros((N, N), dtype=float)
    leaves = list(BROADCAST_LEAVES)

    for idx, leaf in enumerate(leaves):
        prev_leaf = leaves[(idx - 1) % len(leaves)]
        next_leaf = leaves[(idx + 1) % len(leaves)]
        if kind == "dual":
            raw[leaf, 0] = 0.50 * h
            raw[leaf, 1] = 0.50 * h
        elif kind == "split":
            primary = 0 if leaf in BROADCAST_GROUP_A else 1
            secondary = 1 if primary == 0 else 0
            raw[leaf, primary] = 0.85 * h
            raw[leaf, secondary] = 0.15 * h
        elif kind == "asymmetric":
            raw[leaf, 0] = 0.75 * h
            raw[leaf, 1] = 0.25 * h
        else:
            raise ValueError(kind)
        remaining = 1.0 - h
        raw[leaf, prev_leaf] += 0.50 * remaining
        raw[leaf, next_leaf] += 0.50 * remaining

    raw[0, 1] = 0.20
    raw[1, 0] = 0.20
    for leaf in BROADCAST_GROUP_A:
        raw[0, leaf] += 0.80 / len(BROADCAST_GROUP_A)
    for leaf in BROADCAST_GROUP_B:
        raw[1, leaf] += 0.80 / len(BROADCAST_GROUP_B)

    rng = np.random.default_rng(
        30_000 + 100 * list(BROADCAST_FAMILIES).index(family_name) + int(environment_seed)
    )
    raw = _jitter_positive_edges(raw, rng, EDGE_WEIGHT_JITTER_FRAC)
    return raw, _normalize_rows(raw)


def build_broadcast_x0(*, hub_level: float, leaf_level: float, environment_seed: int) -> np.ndarray:
    x0 = np.empty(N, dtype=float)
    x0[0] = float(hub_level) - 0.01
    x0[1] = float(hub_level) + 0.01
    offsets = np.linspace(-0.035, 0.035, len(BROADCAST_LEAVES))
    x0[list(BROADCAST_LEAVES)] = float(leaf_level) + offsets
    rng = np.random.default_rng(40_000 + int(environment_seed))
    x0 += rng.uniform(-BROADCAST_OPINION_JITTER, BROADCAST_OPINION_JITTER, size=N)
    return np.clip(x0, 0.0, 1.0)


In [4]:
def edge_average_precision(
    A_true: np.ndarray,
    A_hat: np.ndarray,
    tol: float = 1e-12,
) -> float:
    A_true = np.asarray(A_true, dtype=float)
    A_hat = np.asarray(A_hat, dtype=float)
    n = A_true.shape[0]
    mask = ~np.eye(n, dtype=bool)
    y = (A_true[mask] > tol).astype(int)
    scores = A_hat[mask]
    positives = int(y.sum())
    if positives == 0:
        return np.nan
    order = np.argsort(-scores, kind="mergesort")
    y_sorted = y[order]
    tp = np.cumsum(y_sorted)
    precision = tp / (np.arange(len(y_sorted)) + 1)
    return float(np.sum(precision * y_sorted) / positives)


def support_topdegree_recall(
    A_true: np.ndarray,
    A_hat: np.ndarray,
    tol: float = 1e-12,
) -> float:
    vals = []
    A_true = np.asarray(A_true, dtype=float)
    A_hat = np.asarray(A_hat, dtype=float)
    for i in range(A_true.shape[0]):
        true_support = set(np.flatnonzero(A_true[i] > tol).tolist())
        true_support.discard(i)
        d = len(true_support)
        if d == 0:
            continue
        scores = A_hat[i].copy()
        scores[i] = -np.inf
        pred = set(np.argsort(-scores, kind="mergesort")[:d].tolist())
        vals.append(len(true_support & pred) / d)
    return float(np.mean(vals)) if vals else np.nan


def true_support_mass(
    A_true: np.ndarray,
    A_hat: np.ndarray,
    tol: float = 1e-12,
) -> float:
    vals = []
    A_true = np.asarray(A_true, dtype=float)
    A_hat = np.asarray(A_hat, dtype=float)
    for i in range(A_true.shape[0]):
        support = A_true[i] > tol
        support[i] = False
        vals.append(float(A_hat[i, support].sum()))
    return float(np.mean(vals))


def recovery_metrics(
    A_true: np.ndarray,
    A_hat: np.ndarray,
    v_true: np.ndarray,
) -> dict[str, float]:
    A_true = np.asarray(A_true, dtype=float)
    A_hat = np.asarray(A_hat, dtype=float)
    diff = A_hat - A_true
    v_hat = centrality_from_A(A_hat)
    row_tv = 0.5 * np.abs(diff).sum(axis=1)

    eps = 1e-12
    P = np.clip(A_hat, eps, None)
    row_entropy = -np.sum(P * np.log(P), axis=1)
    denom = np.log(max(2, A_hat.shape[1] - 1))
    row_entropy_norm = row_entropy / denom
    effective_neighbors = np.exp(row_entropy)

    return {
        "A_MAE": float(np.mean(np.abs(diff))),
        "A_Fro": float(np.linalg.norm(diff, ord="fro")),
        "row_TV": float(np.mean(row_tv)),
        "true_edge_mass": true_support_mass(A_true, A_hat),
        "support_topdegree_recall": support_topdegree_recall(A_true, A_hat),
        "edge_AP": edge_average_precision(A_true, A_hat),
        "centrality_L1": float(
            np.sum(np.abs(v_hat - np.asarray(v_true, dtype=float)))
        ),
        "row_entropy_norm": float(np.mean(row_entropy_norm)),
        "effective_neighbors": float(np.mean(effective_neighbors)),
    }


## Build the complete fixed environment registry


In [5]:
ENVIRONMENT_SPECS: list[dict[str, Any]] = []

for dynamics, dynamics_params in UNSTRUCTURED_DYNAMICS.items():
    for topology_seed in UNSTRUCTURED_TOPOLOGY_SEEDS:
        for opinion_seed in UNSTRUCTURED_OPINION_SEEDS:
            A = np.asarray(UNSTRUCTURED_A_BY_SEED[int(topology_seed)], dtype=float).copy()
            x0 = build_unstructured_x0(int(opinion_seed))
            ENVIRONMENT_SPECS.append({
                "environment_index": len(ENVIRONMENT_SPECS),
                "environment_id": f"unstructured__{dynamics}__topo{int(topology_seed):02d}__x{int(opinion_seed):02d}",
                "scenario_class": "unstructured_random",
                "family": "barabasi_albert_spread",
                "dynamics": dynamics,
                "topology_seed": int(topology_seed),
                "opinion_seed": int(opinion_seed),
                "environment_seed": None,
                "dynamics_params": dict(dynamics_params),
                "A": A,
                "x0": x0,
                "v_true": centrality_from_A(A),
            })

for family in HK_FAMILIES:
    for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
        raw, A = build_hk_graph(family, environment_seed)
        x0 = build_hk_x0(environment_seed)
        ENVIRONMENT_SPECS.append({
            "environment_index": len(ENVIRONMENT_SPECS),
            "environment_id": f"structured__hegselmannkrause__{family}__env{environment_seed:02d}",
            "scenario_class": "structured_mechanism",
            "family": family,
            "dynamics": "hegselmannkrause",
            "topology_seed": None,
            "opinion_seed": None,
            "environment_seed": int(environment_seed),
            "dynamics_params": {"hk_epsilon": STRUCTURED_HK_EPSILON, "hk_include_self": True},
            "raw": raw,
            "A": A,
            "x0": x0,
            "v_true": centrality_from_A(A),
        })

for dynamics in ["friedkinjohnsen", "nonlinearinfluence"]:
    for family in BROADCAST_FAMILIES:
        for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
            raw, A = build_broadcaster_graph(family, environment_seed)
            if dynamics == "friedkinjohnsen":
                x0 = build_broadcast_x0(
                    hub_level=FJ_HUB_LEVEL,
                    leaf_level=FJ_LEAF_LEVEL,
                    environment_seed=environment_seed,
                )
                dynamics_params = {"fj_lambda": STRUCTURED_FJ_LAMBDA}
            else:
                x0 = build_broadcast_x0(
                    hub_level=NLI_HUB_LEVEL,
                    leaf_level=NLI_LEAF_LEVEL,
                    environment_seed=environment_seed,
                )
                dynamics_params = {"nonlinear_beta": STRUCTURED_NLI_BETA}
            ENVIRONMENT_SPECS.append({
                "environment_index": len(ENVIRONMENT_SPECS),
                "environment_id": f"structured__{dynamics}__{family}__env{environment_seed:02d}",
                "scenario_class": "structured_mechanism",
                "family": family,
                "dynamics": dynamics,
                "topology_seed": None,
                "opinion_seed": None,
                "environment_seed": int(environment_seed),
                "dynamics_params": dynamics_params,
                "raw": raw,
                "A": A,
                "x0": x0,
                "v_true": centrality_from_A(A),
            })

assert len(ENVIRONMENT_SPECS) == 195
registry_df = pd.DataFrame([
    {k: spec.get(k) for k in [
        "environment_index", "environment_id", "scenario_class", "family", "dynamics",
        "topology_seed", "opinion_seed", "environment_seed"
    ]}
    for spec in ENVIRONMENT_SPECS
])
print(registry_df.groupby(["scenario_class", "dynamics"]).size().to_string())
print("Distinct environments:", len(ENVIRONMENT_SPECS))
print("Learned evaluations:", len(ENVIRONMENT_SPECS) * len(LEARNING_SEEDS))


scenario_class        dynamics          
structured_mechanism  friedkinjohnsen       15
                      hegselmannkrause      15
                      nonlinearinfluence    15
unstructured_random   coca                  25
                      friedkinjohnsen       25
                      hegselmannkrause      25
                      laplacian             25
                      nonlinearinfluence    25
                      repulsion             25
Distinct environments: 195
Learned evaluations: 585


## Environment construction, abundant collection, fitting, and frozen evaluation


In [6]:
def make_env(
    spec: dict[str, Any],
    *,
    seed: int,
    initial_opinions: np.ndarray | None = None,
) -> NetworkGraph:
    params = dict(spec["dynamics_params"])
    x_init = np.asarray(spec["x0"] if initial_opinions is None else initial_opinions, dtype=float).copy()
    kwargs = dict(
        connectivity_matrix=np.asarray(spec["A"], dtype=float).copy(),
        num_agents=N,
        max_u=MAX_U,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        initial_opinions=x_init,
        control_resistance=np.zeros(N, dtype=float),
        dynamics_model=spec["dynamics"],
        budget=1000.0,
        max_steps=max(NUM_CAMPAIGNS, ABUNDANT_CAMPAIGNS_PER_RESET) + 20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        terminate_when_converged=False,
        seed=int(seed),
    )
    if spec["dynamics"] == "hegselmannkrause":
        kwargs["hk_epsilon"] = float(params["hk_epsilon"])
        kwargs["hk_include_self"] = bool(params["hk_include_self"])
    elif spec["dynamics"] == "friedkinjohnsen":
        kwargs["fj_lambda"] = float(params["fj_lambda"])
        # Keep prejudice fixed to the paper environment while training from many
        # different observed states. This is the same fixed FJ environment.
        kwargs["fj_prejudice"] = np.asarray(spec["x0"], dtype=float).copy()
    elif spec["dynamics"] == "nonlinearinfluence":
        kwargs["nonlinear_beta"] = float(params["nonlinear_beta"])
    elif spec["dynamics"] == "repulsion":
        kwargs["repulsion_epsilon"] = float(params["repulsion_epsilon"])
        kwargs["repulsion_strength"] = float(params["repulsion_strength"])
    return NetworkGraph(**kwargs)


def set_state(env: NetworkGraph, x0: np.ndarray) -> None:
    env.reset()
    env.opinions = np.asarray(x0, dtype=float).copy()
    if hasattr(env, "state"):
        try:
            env.state = np.asarray(x0, dtype=float).copy()
        except Exception:
            pass


def uniform_action(max_u: np.ndarray, budget: float) -> np.ndarray:
    return online_single_shot.uniform_budget_action(np.asarray(max_u, dtype=float).reshape(-1), float(budget))


def true_graph_action(env: NetworkGraph, v_true: np.ndarray) -> np.ndarray:
    action, _ = centrality_based_continuous_control(
        env, float(B_CAMPAIGN), v=np.asarray(v_true, dtype=float)
    )
    return np.asarray(action, dtype=float)


def abundant_reset_x0(spec: dict[str, Any], rng: np.random.Generator) -> np.ndarray:
    # Draw fresh initial conditions from the same distribution/family used
    # to define the environment, rather than forcing every reset to use the
    # exact evaluation x0 or applying the generic [0,1] permutation to all cases.
    if spec["scenario_class"] == "unstructured_random":
        return rng.permutation(np.linspace(0.0, 1.0, N)).astype(float)

    if spec["dynamics"] == "hegselmannkrause":
        base = np.array(
            [0.35, 0.27, 0.28, 0.29, 0.30, 0.31, 0.32,
             0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.70, 0.71],
            dtype=float,
        )
        return np.clip(
            base + rng.uniform(-HK_OPINION_JITTER, HK_OPINION_JITTER, size=N),
            0.0, 1.0,
        )

    if spec["dynamics"] == "friedkinjohnsen":
        hub_level, leaf_level = FJ_HUB_LEVEL, FJ_LEAF_LEVEL
    elif spec["dynamics"] == "nonlinearinfluence":
        hub_level, leaf_level = NLI_HUB_LEVEL, NLI_LEAF_LEVEL
    else:
        raise ValueError(
            f"Unexpected structured dynamics for abundant x0 sampling: {spec['dynamics']}"
        )

    x0 = np.empty(N, dtype=float)
    x0[0] = float(hub_level) - 0.01
    x0[1] = float(hub_level) + 0.01
    offsets = np.linspace(-0.035, 0.035, len(BROADCAST_LEAVES))
    x0[list(BROADCAST_LEAVES)] = float(leaf_level) + offsets
    x0 += rng.uniform(
        -BROADCAST_OPINION_JITTER,
        BROADCAST_OPINION_JITTER,
        size=N,
    )
    return np.clip(x0, 0.0, 1.0)


def collect_abundant_dataset(spec: dict[str, Any]) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    data_seed = DATA_SEED_BASE + 10_000 * int(spec["environment_index"])
    rng = np.random.default_rng(data_seed)
    buf_x: list[np.ndarray] = []
    buf_y: list[np.ndarray] = []
    step_calls = 0
    t0 = time.perf_counter()

    for reset_idx in range(ABUNDANT_RESETS):
        x0_train = abundant_reset_x0(spec, rng)
        env = make_env(spec, seed=data_seed + reset_idx, initial_opinions=x0_train)
        set_state(env, x0_train)
        max_u_vec = np.asarray(env.max_u, dtype=float).reshape(-1)

        for campaign in range(ABUNDANT_CAMPAIGNS_PER_RESET):
            if campaign == 0:
                action = np.zeros(N, dtype=float)
            else:
                random_scores = rng.random(N)
                action = online_single_shot.waterfill_from_scores(
                    random_scores,
                    max_u=max_u_vec,
                    budget=float(B_CAMPAIGN),
                )
            _, _, done, trunc, info = env.step(action)
            step_calls += 1
            inter = info.get("intermediate_states")
            if inter is None:
                raise RuntimeError("env.step did not return info['intermediate_states']")
            Xp, Yp = pairs_from_intermediate(np.asarray(inter, dtype=float))
            buf_x.append(np.asarray(Xp, dtype=float))
            buf_y.append(np.asarray(Yp, dtype=float))
            if done or trunc:
                break

    X = np.concatenate(buf_x, axis=0)
    Y = np.concatenate(buf_y, axis=0)
    return X, Y, {
        "n_pairs": int(X.shape[0]),
        "n_resets": int(ABUNDANT_RESETS),
        "campaigns_per_reset": int(ABUNDANT_CAMPAIGNS_PER_RESET),
        "env_step_calls": int(step_calls),
        "collection_elapsed_s": float(time.perf_counter() - t0),
        "data_seed": int(data_seed),
    }


def seed_everything(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def fit_abundant_identifier(
    X: np.ndarray,
    Y: np.ndarray,
    *,
    learning_seed: int,
    regularizer_kind: str,
    regularizer_lambda: float,
) -> tuple[GraphIdentifierEnv, np.ndarray, dict[str, Any]]:
    seed_everything(learning_seed)
    with contextlib.redirect_stdout(io.StringIO()):
        model = GraphIdentifierEnv(
            N=N,
            s=T_S,
            regularizer_kind=str(regularizer_kind),
            regularizer_lambda=float(regularizer_lambda),
            zero_diag=True,
            **IDENTIFIER_KWARGS,
        )
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        A_hat = train_graph_identifier(
            model,
            X,
            Y,
            lr=FIT_LR,
            batch_size=FIT_BATCH_SIZE,
            max_steps=FIT_MAX_STEPS,
            mae_stop=FIT_MAE_STOP,
            device=DEVICE,
            fit_check_every=FIT_CHECK_EVERY,
            verbose_every=0,
        )
    fit_elapsed = time.perf_counter() - t0
    model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        Yt = torch.tensor(Y, dtype=torch.float32, device=DEVICE)
        pred = model.predict_next(Xt)
        train_mae = float((pred - Yt).abs().mean().cpu().item())
        identity_mae = float((Xt - Yt).abs().mean().cpu().item())
        reg_value = float(model.regularizer_value().cpu().item())
    return model, np.asarray(A_hat, dtype=float), {
        "fit_elapsed_s": float(fit_elapsed),
        "train_mae": train_mae,
        "identity_mae": identity_mae,
        "model_over_identity": float(train_mae / identity_mae) if identity_mae > 1e-12 else np.nan,
        "regularizer_value": reg_value,
        "regularizer_weighted": float(regularizer_lambda) * reg_value,
        "steps_run": int(getattr(model, "last_fit_info", {}).get("steps_run", FIT_MAX_STEPS)),
        "stop_reason": str(getattr(model, "last_fit_info", {}).get("stop_reason", "unknown")),
    }

def rollout_fixed_policy(spec: dict[str, Any], *, policy: str) -> dict[str, Any]:
    env = make_env(spec, seed=500_000 + int(spec["environment_index"]))
    set_state(env, spec["x0"])
    states = [np.asarray(spec["x0"], dtype=float).copy()]
    actions, rewards, intermediate_states_list = [], [], []
    max_u_vec = np.asarray(env.max_u, dtype=float).reshape(-1)
    for campaign in range(NUM_CAMPAIGNS):
        if campaign == 0 or policy == "no_control":
            action = np.zeros(N, dtype=float)
        elif policy == "uniform":
            action = uniform_action(max_u_vec, B_CAMPAIGN)
        elif policy == "true_graph":
            action = true_graph_action(env, spec["v_true"])
        else:
            raise ValueError(policy)
        x_next, reward, done, trunc, info = env.step(action)
        states.append(np.asarray(x_next, dtype=float).copy())
        actions.append(np.asarray(action, dtype=float).copy())
        rewards.append(float(reward))
        intermediate_states_list.append(np.asarray(info["intermediate_states"], dtype=float).copy())
        if done or trunc:
            break
    return {
        "policy": policy,
        "states": np.asarray(states, dtype=float),
        "actions": np.asarray(actions, dtype=float),
        "rewards": np.asarray(rewards, dtype=float),
        "intermediate_states_list": intermediate_states_list,
    }


def rollout_frozen_learned(spec: dict[str, Any], model: GraphIdentifierEnv) -> dict[str, Any]:
    env = make_env(spec, seed=700_000 + int(spec["environment_index"]))
    set_state(env, spec["x0"])
    state = np.asarray(spec["x0"], dtype=float).copy()
    states = [state.copy()]
    actions, rewards, intermediate_states_list = [], [], []
    max_u_vec = np.asarray(env.max_u, dtype=float).reshape(-1)

    for campaign in range(NUM_CAMPAIGNS):
        if campaign == 0:
            action = np.zeros(N, dtype=float)
        else:
            scores, _, _ = online_single_shot.learned_lambda_mix_scores(
                model,
                state,
                desired_opinion=TARGET,
                lambda_mix=LAMBDA_MIX,
                device=DEVICE,
            )
            action = online_single_shot.waterfill_from_scores(
                scores,
                max_u=max_u_vec,
                budget=float(B_CAMPAIGN),
            )
        x_next, reward, done, trunc, info = env.step(action)
        state = np.asarray(x_next, dtype=float).copy()
        states.append(state.copy())
        actions.append(np.asarray(action, dtype=float).copy())
        rewards.append(float(reward))
        intermediate_states_list.append(np.asarray(info["intermediate_states"], dtype=float).copy())
        if done or trunc:
            break

    return {
        "policy": "learned_abundant_nonlinear",
        "states": np.asarray(states, dtype=float),
        "actions": np.asarray(actions, dtype=float),
        "rewards": np.asarray(rewards, dtype=float),
        "intermediate_states_list": intermediate_states_list,
    }


## Metrics and persistent caches


In [7]:
def base_metrics(rollout: dict[str, Any]) -> dict[str, Any]:
    states = np.asarray(rollout["states"], dtype=float)
    mean_path = states.mean(axis=1)
    distance_path = np.mean(np.abs(TARGET - states), axis=1)
    return {
        "mean_start": float(mean_path[0]),
        "mean_end": float(mean_path[-1]),
        "mean_over_trajectory": float(mean_path.mean()),
        "min_end": float(states[-1].min()),
        "max_end": float(states[-1].max()),
        "final_mean_abs_distance_to_target": float(distance_path[-1]),
        "trajectory_mean_abs_distance_to_target": float(distance_path.mean()),
        "n_eval_campaigns": int(len(states) - 1),
    }


def mechanism_metrics(spec: dict[str, Any], rollout: dict[str, Any]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    actions = np.asarray(rollout["actions"], dtype=float)
    if spec["scenario_class"] == "structured_mechanism" and len(actions) > 1:
        hubs = [HK_LOW_HUB, HK_HIGH_HUB] if spec["dynamics"] == "hegselmannkrause" else list(BROADCAST_HUBS)
        controlled = actions[1:]
        total = controlled.sum(axis=1)
        hub_total = controlled[:, hubs].sum(axis=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            fractions = np.where(total > 1e-12, hub_total / total, np.nan)
        result["mean_controlled_hub_budget_fraction"] = float(np.nanmean(fractions))
    return result


def summarize_rollout(spec: dict[str, Any], rollout: dict[str, Any]) -> dict[str, Any]:
    return {**base_metrics(rollout), **mechanism_metrics(spec, rollout)}


def common_row(spec: dict[str, Any]) -> dict[str, Any]:
    return {
        "environment_id": spec["environment_id"],
        "environment_index": int(spec["environment_index"]),
        "scenario_class": spec["scenario_class"],
        "family": spec["family"],
        "dynamics": spec["dynamics"],
        "topology_seed": spec.get("topology_seed"),
        "opinion_seed": spec.get("opinion_seed"),
        "environment_seed": spec.get("environment_seed"),
    }



SCIENTIFIC_CONFIG = {
    "pipeline_version": PIPELINE_VERSION,
    "N": N,
    "num_campaigns": NUM_CAMPAIGNS,
    "t_campaign": T_CAMPAIGN,
    "t_s": T_S,
    "max_u": MAX_U,
    "total_controlled_budget": TOTAL_CONTROLLED_BUDGET,
    "B_campaign": B_CAMPAIGN,
    "lambda_mix": LAMBDA_MIX,
    "variants": VARIANTS,
    "fit_lr": FIT_LR,
    "fit_max_steps": FIT_MAX_STEPS,
    "fit_mae_stop": FIT_MAE_STOP,
    "fit_batch_size": FIT_BATCH_SIZE,
    "fit_check_every": FIT_CHECK_EVERY,
    "identifier_kwargs": IDENTIFIER_KWARGS,
    "learning_seeds": LEARNING_SEEDS,
    "abundant_resets": ABUNDANT_RESETS,
    "abundant_campaigns_per_reset": ABUNDANT_CAMPAIGNS_PER_RESET,
    "collection_policy": COLLECTION_POLICY,
    "collection_x0": COLLECTION_X0,
    "data_seed_base": DATA_SEED_BASE,
    "source_study": SOURCE_STUDY,
    "unstructured_dynamics": UNSTRUCTURED_DYNAMICS,
    "unstructured_topology_seeds": UNSTRUCTURED_TOPOLOGY_SEEDS,
    "unstructured_opinion_seeds": UNSTRUCTURED_OPINION_SEEDS,
    "structured_environment_seeds": STRUCTURED_ENVIRONMENT_SEEDS,
    "hk_families": HK_FAMILIES,
    "broadcast_families": BROADCAST_FAMILIES,
}
CONFIG_HASH = hashlib.sha256(
    json.dumps(SCIENTIFIC_CONFIG, sort_keys=True).encode()
).hexdigest()[:16]
print("Config hash:", CONFIG_HASH)


def environment_cache_dir(spec: dict[str, Any]) -> Path:
    return CACHE_ROOT / CONFIG_HASH / spec["environment_id"]


def _atomic_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=str),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def _save_npz_atomic(path: Path, **arrays: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with tmp.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(tmp, path)


def _save_torch_atomic(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def _valid_dataset_meta(meta: dict[str, Any]) -> bool:
    return (
        int(meta.get("n_resets", -1)) == int(ABUNDANT_RESETS)
        and int(meta.get("campaigns_per_reset", -1))
        == int(ABUNDANT_CAMPAIGNS_PER_RESET)
    )


_SOURCE_ENV_DIR_CACHE: dict[str, Path | None] = {}


def locate_source_environment_dir(spec: dict[str, Any]) -> Path | None:
    env_id = spec["environment_id"]
    if env_id in _SOURCE_ENV_DIR_CACHE:
        return _SOURCE_ENV_DIR_CACHE[env_id]

    hit = None
    if SOURCE_CACHE_STUDY_ROOT.exists():
        for config_dir in sorted(SOURCE_CACHE_STUDY_ROOT.iterdir()):
            d = config_dir / env_id
            meta_path = d / "dataset_meta.json"
            data_path = d / "dataset.npz"
            if not (meta_path.exists() and data_path.exists()):
                continue
            try:
                meta = json.loads(meta_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if _valid_dataset_meta(meta):
                hit = d
                break

    _SOURCE_ENV_DIR_CACHE[env_id] = hit
    return hit


def current_dataset_cache_complete(spec: dict[str, Any]) -> bool:
    d = environment_cache_dir(spec)
    return (d / "dataset.npz").exists() and (d / "dataset_meta.json").exists()


def ensure_dataset(
    spec: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    # 1) Current experiment cache.
    d = environment_cache_dir(spec)
    if current_dataset_cache_complete(spec):
        with np.load(d / "dataset.npz") as arrays:
            X = np.asarray(arrays["X"], dtype=float)
            Y = np.asarray(arrays["Y"], dtype=float)
        meta = json.loads(
            (d / "dataset_meta.json").read_text(encoding="utf-8")
        )
        return X, Y, meta

    # 2) Reuse the exact corrected 25x20 dataset from the no-reg run.
    source_d = locate_source_environment_dir(spec)
    if source_d is not None:
        with np.load(source_d / "dataset.npz") as arrays:
            X = np.asarray(arrays["X"], dtype=float)
            Y = np.asarray(arrays["Y"], dtype=float)
        meta = json.loads(
            (source_d / "dataset_meta.json").read_text(encoding="utf-8")
        )
        meta = {
            **meta,
            "dataset_origin": "reused_25x20_no_reg",
            "dataset_source_dir": str(source_d),
        }
        return X, Y, meta

    # 3) Safe fallback if the no-reg 25x20 run has not produced this dataset.
    X, Y, meta = collect_abundant_dataset(spec)
    meta = {**meta, "dataset_origin": "collected_in_paired_run"}
    _save_npz_atomic(d / "dataset.npz", X=X, Y=Y)
    _atomic_json(
        d / "dataset_meta.json",
        {**meta, "config_hash": CONFIG_HASH},
    )
    return X, Y, meta


_SOURCE_SUMMARY = None
if SOURCE_RESULTS_DIR.exists() and (SOURCE_RESULTS_DIR / "combined_summary.csv").exists():
    try:
        _SOURCE_SUMMARY = pd.read_csv(SOURCE_RESULTS_DIR / "combined_summary.csv")
        print("Reusable no-reg summary rows:", len(_SOURCE_SUMMARY))
    except Exception:
        _SOURCE_SUMMARY = None


def source_no_reg_row(
    spec: dict[str, Any],
    learning_seed: int,
) -> dict[str, Any] | None:
    if _SOURCE_SUMMARY is None:
        return None
    x = _SOURCE_SUMMARY[
        (_SOURCE_SUMMARY["environment_id"] == spec["environment_id"])
        & (_SOURCE_SUMMARY["policy"] == "learned_abundant_nonlinear")
        & (_SOURCE_SUMMARY["learning_seed"] == int(learning_seed))
    ]
    if len(x) != 1:
        return None
    row = x.iloc[0].to_dict()
    row.update({
        "policy": "learned_abundant_none",
        "regularizer_kind": "none",
        "regularizer_lambda": 0.0,
        "cache_origin": "reused_25x20_no_reg",
    })
    return row


def baseline_cache_complete(spec: dict[str, Any]) -> bool:
    d = environment_cache_dir(spec)
    return (d / "baseline_summary.json").exists()


def ensure_baseline_cache(spec: dict[str, Any]) -> list[dict[str, Any]]:
    d = environment_cache_dir(spec)
    if baseline_cache_complete(spec):
        return json.loads(
            (d / "baseline_summary.json").read_text(encoding="utf-8")
        )
    rows = []
    for policy in ["no_control", "uniform", "true_graph"]:
        rollout = rollout_fixed_policy(spec, policy=policy)
        rows.append({
            **common_row(spec),
            "policy": policy,
            "learning_seed": None,
            "regularizer_kind": None,
            "regularizer_lambda": np.nan,
            "data_regime": "baseline",
            **summarize_rollout(spec, rollout),
        })
    _atomic_json(d / "baseline_summary.json", rows)
    return rows


def variant_tag(kind: str, lam: float, learning_seed: int) -> str:
    lam_tag = f"{float(lam):.0e}".replace("+", "")
    return f"{kind}__lambda_{lam_tag}__seed_{int(learning_seed):02d}"


def learned_cache_paths(
    spec: dict[str, Any],
    *,
    kind: str,
    lam: float,
    learning_seed: int,
) -> tuple[Path, Path, Path]:
    d = environment_cache_dir(spec)
    tag = variant_tag(kind, lam, learning_seed)
    return (
        d / f"{tag}_summary.json",
        d / f"{tag}_rollout.npz",
        d / f"{tag}_model_state.pt",
    )


def learned_cache_complete(
    spec: dict[str, Any],
    *,
    kind: str,
    lam: float,
    learning_seed: int,
) -> bool:
    summary_path, rollout_path, state_path = learned_cache_paths(
        spec,
        kind=kind,
        lam=lam,
        learning_seed=learning_seed,
    )
    if kind == "none":
        # A reused no-reg row is sufficient; model artifacts already live in
        # the source 25x20 experiment.
        return summary_path.exists()
    return (
        summary_path.exists()
        and rollout_path.exists()
        and state_path.exists()
    )


def ensure_learned_cache(
    spec: dict[str, Any],
    *,
    kind: str,
    lam: float,
    learning_seed: int,
    X: np.ndarray,
    Y: np.ndarray,
    data_meta: dict[str, Any],
) -> dict[str, Any]:
    summary_path, rollout_path, state_path = learned_cache_paths(
        spec,
        kind=kind,
        lam=lam,
        learning_seed=learning_seed,
    )

    if learned_cache_complete(
        spec,
        kind=kind,
        lam=lam,
        learning_seed=learning_seed,
    ):
        return json.loads(summary_path.read_text(encoding="utf-8"))

    # Reuse the already-computed 25x20 no-reg result if available.
    if kind == "none" and float(lam) == 0.0:
        reused = source_no_reg_row(spec, learning_seed)
        if reused is not None:
            reused["dataset_origin"] = data_meta.get(
                "dataset_origin",
                "reused_25x20_no_reg",
            )
            _atomic_json(summary_path, reused)
            return reused

    model, A_hat, fit_info = fit_abundant_identifier(
        X,
        Y,
        learning_seed=learning_seed,
        regularizer_kind=kind,
        regularizer_lambda=lam,
    )
    rollout = rollout_frozen_learned(spec, model)
    v_hat_static = centrality_from_A(A_hat)

    policy = (
        "learned_abundant_none"
        if kind == "none"
        else f"learned_abundant_{kind}_{float(lam):.0e}"
    )

    row = {
        **common_row(spec),
        "policy": policy,
        "learning_seed": int(learning_seed),
        "regularizer_kind": str(kind),
        "regularizer_lambda": float(lam),
        "cache_origin": "fitted_in_paired_run",
        "dataset_origin": data_meta.get(
            "dataset_origin",
            "unknown",
        ),
        "data_regime": "abundant_prior_frozen",
        "abundant_resets": int(ABUNDANT_RESETS),
        "abundant_campaigns_per_reset": int(
            ABUNDANT_CAMPAIGNS_PER_RESET
        ),
        "train_pairs_total": int(data_meta["n_pairs"]),
        "data_collection_elapsed_s": float(
            data_meta.get("collection_elapsed_s", np.nan)
        ),
        "fit_elapsed_s": float(fit_info["fit_elapsed_s"]),
        "fit_steps_run": int(fit_info["steps_run"]),
        "fit_stop_reason": fit_info["stop_reason"],
        "train_mae": float(fit_info["train_mae"]),
        "identity_mae": float(fit_info["identity_mae"]),
        "model_over_identity": float(
            fit_info["model_over_identity"]
        ),
        "regularizer_value": float(
            fit_info["regularizer_value"]
        ),
        "regularizer_weighted": float(
            fit_info["regularizer_weighted"]
        ),
        **recovery_metrics(
            np.asarray(spec["A"], dtype=float),
            np.asarray(A_hat, dtype=float),
            np.asarray(spec["v_true"], dtype=float),
        ),
        "v_static_L1": float(
            np.sum(
                np.abs(
                    v_hat_static
                    - np.asarray(spec["v_true"], dtype=float)
                )
            )
        ),
        **summarize_rollout(spec, rollout),
    }

    artifact_arrays: dict[str, np.ndarray] = {
        "states": np.asarray(rollout["states"], dtype=float),
        "actions": np.asarray(rollout["actions"], dtype=float),
        "rewards": np.asarray(rollout["rewards"], dtype=float),
        "A_hat": np.asarray(A_hat, dtype=float),
        "v_hat_static": np.asarray(v_hat_static, dtype=float),
    }
    for campaign, inter in enumerate(
        rollout["intermediate_states_list"]
    ):
        artifact_arrays[
            f"intermediate__{campaign:02d}"
        ] = np.asarray(inter, dtype=float)

    _save_npz_atomic(rollout_path, **artifact_arrays)
    _save_torch_atomic(
        state_path,
        {
            "state_dict": model.state_dict(),
            "identifier_kwargs": IDENTIFIER_KWARGS,
            "N": N,
            "s": T_S,
            "lambda_mix": LAMBDA_MIX,
            "regularizer_kind": str(kind),
            "regularizer_lambda": float(lam),
            "learning_seed": int(learning_seed),
            "config_hash": CONFIG_HASH,
        },
    )
    _atomic_json(summary_path, row)
    return row


Config hash: 5eaa1f04f5d45551


## Work estimate and parallel worker


In [8]:

N_ENV = len(ENVIRONMENT_SPECS)
N_VARIANT_RUNS = (
    N_ENV * len(LEARNING_SEEDS) * len(VARIANTS)
)
ABUNDANT_COLLECTION_ENV_STEPS = (
    N_ENV
    * ABUNDANT_RESETS
    * ABUNDANT_CAMPAIGNS_PER_RESET
)

print("Expected environments:", N_ENV)
print("Paired learned rows:", N_VARIANT_RUNS)
print("Variants:", VARIANTS)
print(
    "Abundant collection environment steps:",
    ABUNDANT_COLLECTION_ENV_STEPS,
)
print("Approx environments/shard:", N_ENV / NUM_SHARDS)


def run_worker() -> None:
    assigned_specs = [
        spec
        for spec in ENVIRONMENT_SPECS
        if int(spec["environment_index"]) % NUM_SHARDS == SHARD_ID
    ]
    print("Assigned environments:", len(assigned_specs))
    shard_rows: list[dict[str, Any]] = []
    t0 = time.perf_counter()

    for position, spec in enumerate(assigned_specs, start=1):
        env_t0 = time.perf_counter()
        print(
            f"\n[{position}/{len(assigned_specs)}] "
            f"{spec['environment_id']}",
            flush=True,
        )

        baseline_rows = ensure_baseline_cache(spec)
        X, Y, data_meta = ensure_dataset(spec)
        print(
            f"  dataset: {len(X)} pairs | "
            f"origin={data_meta.get('dataset_origin', 'current_cache')}",
            flush=True,
        )
        shard_rows.extend(baseline_rows)

        for variant in VARIANTS:
            kind = variant["regularizer_kind"]
            lam = float(variant["regularizer_lambda"])
            for learning_seed in LEARNING_SEEDS:
                row = ensure_learned_cache(
                    spec,
                    kind=kind,
                    lam=lam,
                    learning_seed=learning_seed,
                    X=X,
                    Y=Y,
                    data_meta=data_meta,
                )
                shard_rows.append(row)
                print(
                    f"  {kind} {lam:g} seed {learning_seed}: "
                    f"origin={row.get('cache_origin')} "
                    f"train_mae={float(row['train_mae']):.3g} "
                    f"mean_end={float(row['mean_end']):.4f}",
                    flush=True,
                )

        env_elapsed = time.perf_counter() - env_t0
        elapsed_so_far = time.perf_counter() - t0
        projected_remaining = (
            elapsed_so_far / position
        ) * (len(assigned_specs) - position)
        print(
            f"  environment elapsed: {env_elapsed:.2f}s",
            flush=True,
        )
        print(
            f"  shard progress: {position}/{len(assigned_specs)} | "
            f"projected remaining ~"
            f"{projected_remaining / 60:.1f} min",
            flush=True,
        )

    elapsed = time.perf_counter() - t0
    shard_df = pd.DataFrame(shard_rows)
    shard_path = (
        RESULTS_DIR / f"worker_shard_{SHARD_ID}_summary.csv"
    )
    shard_df.to_csv(shard_path, index=False)
    _atomic_json(
        RESULTS_DIR / f"SHARD_{SHARD_ID}_COMPLETE.json",
        {
            "status": "success",
            "shard_id": int(SHARD_ID),
            "n_environments": len(assigned_specs),
            "n_rows": len(shard_rows),
            "elapsed_s": float(elapsed),
            "config_hash": CONFIG_HASH,
            "git_commit": GIT_COMMIT,
            "completed_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        },
    )
    print(
        f"\nShard {SHARD_ID} complete "
        f"in {elapsed / 60:.1f} minutes."
    )


if WORKER_MODE:
    run_worker()
else:
    print(
        "Check mode: expensive training is disabled "
        "without ABUNDANT_FULL_SHARD_ID."
    )


Expected environments: 195
Paired learned rows: 1170
Variants: [{'regularizer_kind': 'none', 'regularizer_lambda': 0.0}, {'regularizer_kind': 'gini', 'regularizer_lambda': 0.0001}]
Abundant collection environment steps: 97500
Approx environments/shard: 24.375
Check mode: expensive training is disabled without ABUNDANT_FULL_SHARD_ID.


## Cache completeness and merge


In [9]:

if WORKER_MODE:
    print(
        "Worker mode: skipping global validation and merge. "
        "Run once without ABUNDANT_FULL_SHARD_ID "
        "after all shards finish."
    )
else:
    missing = []
    for spec in ENVIRONMENT_SPECS:
        for variant in VARIANTS:
            for seed in LEARNING_SEEDS:
                if not learned_cache_complete(
                    spec,
                    kind=variant["regularizer_kind"],
                    lam=float(variant["regularizer_lambda"]),
                    learning_seed=seed,
                ):
                    missing.append((
                        spec["environment_id"],
                        variant["regularizer_kind"],
                        variant["regularizer_lambda"],
                        seed,
                    ))

    print("Expected environments:", len(ENVIRONMENT_SPECS))
    print("Expected paired learned rows:", N_VARIANT_RUNS)
    print("Missing learned rows:", len(missing))

    if not missing:
        all_rows: list[dict[str, Any]] = []

        for spec in ENVIRONMENT_SPECS:
            all_rows.extend(ensure_baseline_cache(spec))
            for variant in VARIANTS:
                for seed in LEARNING_SEEDS:
                    summary_path, _, _ = learned_cache_paths(
                        spec,
                        kind=variant["regularizer_kind"],
                        lam=float(variant["regularizer_lambda"]),
                        learning_seed=seed,
                    )
                    all_rows.append(
                        json.loads(
                            summary_path.read_text(
                                encoding="utf-8"
                            )
                        )
                    )

        combined = pd.DataFrame(all_rows)
        combined_path = RESULTS_DIR / "combined_summary.csv"
        combined.to_csv(combined_path, index=False)

        expected_rows = len(ENVIRONMENT_SPECS) * (
            3 + len(VARIANTS) * len(LEARNING_SEEDS)
        )
        assert len(combined) == expected_rows

        learned = combined[
            combined["regularizer_kind"].isin(
                ["none", "gini"]
            )
        ].copy()

        paired = learned.pivot_table(
            index=[
                "environment_id",
                "environment_index",
                "scenario_class",
                "family",
                "dynamics",
                "learning_seed",
            ],
            columns="regularizer_kind",
            values=[
                "mean_end",
                "model_over_identity",
                "row_TV",
                "A_MAE",
                "true_edge_mass",
                "edge_AP",
                "centrality_L1",
                "row_entropy_norm",
                "effective_neighbors",
            ],
            aggfunc="first",
        )
        paired.columns = [
            f"{metric}__{kind}"
            for metric, kind in paired.columns
        ]
        paired = paired.reset_index()
        for metric in [
            "mean_end",
            "model_over_identity",
            "row_TV",
            "A_MAE",
            "true_edge_mass",
            "edge_AP",
            "centrality_L1",
            "row_entropy_norm",
            "effective_neighbors",
        ]:
            paired[f"delta_{metric}_gini_minus_none"] = (
                paired[f"{metric}__gini"]
                - paired[f"{metric}__none"]
            )
        paired.to_csv(
            RESULTS_DIR / "paired_none_vs_gini.csv",
            index=False,
        )

        print("Complete paired cache validated.")
        print("Combined rows:", len(combined))
        print("Combined summary:", combined_path)
        print(
            paired.groupby(
                ["scenario_class", "dynamics"]
            )[
                [
                    "delta_mean_end_gini_minus_none",
                    "delta_row_TV_gini_minus_none",
                    "delta_edge_AP_gini_minus_none",
                ]
            ]
            .mean()
            .round(6)
            .to_string()
        )

        _atomic_json(
            RESULTS_DIR / "COMPUTATION_COMPLETE.json",
            {
                "status": "success",
                "study_name": STUDY_NAME,
                "git_commit": GIT_COMMIT,
                "config_hash": CONFIG_HASH,
                "gini_lambda": GINI_LAMBDA,
                "n_environments": len(ENVIRONMENT_SPECS),
                "n_learning_seeds": len(LEARNING_SEEDS),
                "n_paired_learned_rows": N_VARIANT_RUNS,
                "n_combined_rows": len(combined),
                "completed_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            },
        )
        print("Paired abundant comparison marked complete.")
    else:
        print("First missing:", missing[:10])
        print(
            "Run/re-run the parallel launcher; "
            "completed cache entries are reused."
        )


Expected environments: 195
Expected paired learned rows: 1170
Missing learned rows: 0
Complete paired cache validated.
Combined rows: 1755
Combined summary: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\experiment_2026_10_02_abundant_matched_distribution_25x20_none_vs_gini\combined_summary.csv
                                         delta_mean_end_gini_minus_none  delta_row_TV_gini_minus_none  delta_edge_AP_gini_minus_none
scenario_class       dynamics                                                                                                       
structured_mechanism friedkinjohnsen                           0.000000                     -0.002414                      -0.002674
                     hegselmannkrause                         -0.008819                     -0.000598                      -0.015053
                     nonlinearinfluence                        0.000005                     -0.005396                      -0.038314
unstructu